# NB13A · Recover and resume ATLAS-CONTEXT — revision 2

**31 October 2026. Fixed: missing local NB13 working files.** This notebook is
now self-contained and can run directly with **Run All**. It preserves the same
model, measured protocol and HF namespace; it never starts a fresh fit if a
matching recovery checkpoint is unavailable.

## Kaggle settings

1. Select **GPU T4 ×2** and **Internet ON**.
2. Attach the existing **expanded** `shanmuk4622/tire-dataset-prepared-phase2`
   dataset directly from Kaggle. Do not copy its images to working output.
3. Enable write-capable **HF_TOKEN** in Kaggle Secrets for
   `Shanmuk4622/tyre-wear-study`. Never paste a token in code/output.
4. Start with at least **12 GiB free** working output. Retain the 20 GB output
   budget and automatic 16 GB / free-space guards; extra scratch is not assumed.
5. **Run All**, in order. Do not run another writer for NB13 concurrently.

If the older NB13 session is still accessible, running these cells there keeps
its newer local checkpoint. You may also attach a saved original notebook output
containing the complete `run/` tree; compatible checkpoints are discovered and
copied read-only from that input. Otherwise this notebook restores the full
verified HF snapshot. At the latest independent check, HF held **update 1,294**;
the older notebook printed **2,467**. Unsent progress cannot be reconstructed
unless those older files or a saved output copy still exist.

The recovery reuses the original passed dual-T4/live-HF preflight, verifies the
same source/data/environment contract, restores optimizer/scheduler/scaler/RNG/
sample orders and CSV cursors, and resumes the next completed-update boundary.
It rejects environment or history drift instead of creating another experiment.


In [1]:
import base64,hashlib,importlib.metadata,json,os,subprocess,sys
from pathlib import Path
from filelock import FileLock
assert Path('/kaggle/working').exists(),'Run this notebook on Kaggle.'
WORK=Path('/kaggle/working/tread_two_atlas_context_r3');SOURCE=WORK/'source'
WORK.mkdir(parents=True,exist_ok=True)
os.environ['HF_HUB_DISABLE_IMPLICIT_TOKEN']='1'
os.environ['CUBLAS_WORKSPACE_CONFIG']=':4096:8'
try:version=importlib.metadata.version('huggingface_hub')
except importlib.metadata.PackageNotFoundError:version=None
if version!='0.36.2':subprocess.check_call([sys.executable,'-m','pip','install','--disable-pip-version-check','huggingface_hub==0.36.2'])
payload=json.loads(base64.b64decode('...'))
with FileLock(str(WORK/'NOTEBOOK.lock'),timeout=1):
    for name,value in payload.items():
        path=(WORK/name).resolve();assert path.is_relative_to(WORK.resolve()) and '..' not in Path(name).parts
        body=base64.b64decode(value)
        if path.exists():assert path.read_bytes()==body,'Source or repair mismatch. Preserve existing files; do not replace model/runtime to bypass it.'
        else:path.parent.mkdir(parents=True,exist_ok=True);path.write_bytes(body)
def read(p):return json.loads(p.read_text())
pointer=WORK/'run/POINTER.json'
if pointer.exists():
    p=read(pointer);unit_path=(WORK/'run'/p['unit']).resolve()
    assert unit_path.is_relative_to((WORK/'run').resolve()),'Local unit path escapes this run; preserve it for review.'
    assert hashlib.sha256(unit_path.read_bytes()).hexdigest()==p['sha256'],'Local pointer integrity failed; preserve it.'
    unit=read(unit_path);print('Retained local progress:',json.dumps({k:unit[k] for k in ('seq','updates','phase')}))
else:print('No local checkpoint in this session. Recovery will look for attached saved outputs, otherwise restore verified HF progress. No fresh fit will be started.')
print('Frozen 32-file runtime plus inventory and pinned recovery bootstrap ready. No training has run in this cell.')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 6.0 MB/s  0:00:00
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 1.29.0
    Uninstalling huggingface_hub-1.29.0:
      Successfully uninstalled huggingface_hub-1.29.0


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.26.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.
transformers 5.16.1 requires huggingface-hub<2.0,>=1.5.0, but you have huggingface-hub 0.36.2 which is incompatible.


No local checkpoint in this session. Recovery will look for attached saved outputs, otherwise restore verified HF progress. No fresh fit will be started.
Frozen 32-file runtime plus inventory and pinned recovery bootstrap ready. No training has run in this cell.


## Restore verified progress, then resume the original run

The first stage verifies T4×2, the original environment, expanded Kaggle data
and pinned upstream weights. It reuses the HF-verified gate receipt rather than
repeating a measured model or engineering fit. It prefers a compatible local
or attached saved run, and requires exact history-prefix equality before using it.
Newer local progress is published before fitting continues. With no local run,
the complete HF checkpoint, raw CSVs, artifacts and ledger are downloaded and
verified; this can take time. Cached verified files are reused on retry.

Watch `[NB13 status]` every 15 seconds: operation, local update/stage and free
disk identify verification, download, publication and training. Native state and
all 27 scientific CSV families remain intact. Normal uploads occur every 30
minutes, at major stages and on catchable Stop/error; server backoff may delay
them. Forced kernel death cannot upload unsent progress.

Keep `run/console.log`, `NB13_RECOVERY/STATUS.json`, `EVENTS.jsonl`, and
`THREAD_STACKS.txt` if another error occurs. Environment mismatch gets a separate
JSON report. Do not erase a lease, checkpoint or partial run to bypass a check.
Actual Kaggle execution of this revised recovery remains pending. Save this
executed notebook afterward for an independent HF audit.


In [2]:
import signal,time
from kaggle_secrets import UserSecretsClient
LAUNCHER=WORK/'NB13_RECOVERY/nb13_recovery_launcher_r2.py'
assert hashlib.sha256(LAUNCHER.read_bytes()).hexdigest()=='bc74ea98e637575c2b457e59278da712b07fbc3c5c23c54ff981a2910a4b952b'
with FileLock(str(WORK/'NOTEBOOK.lock'),timeout=1):
    environment=dict(os.environ);environment.pop('HF_TOKEN',None)
    environment['HF_TOKEN']=UserSecretsClient().get_secret('HF_TOKEN')
    try:child=subprocess.Popen([sys.executable,'-B','-u',str(LAUNCHER),'--source',str(SOURCE),'--work',str(WORK),'--contract',str(WORK/'NB13_RECOVERY/PINNED_CONTRACT.json')],env=environment,start_new_session=True)
    finally:environment.pop('HF_TOKEN',None)
    try:
        while True:
            try:result=child.wait(timeout=15);break
            except subprocess.TimeoutExpired:pass
    except KeyboardInterrupt:
        if child.poll() is None:
            os.killpg(child.pid,signal.SIGINT)
            for i in range(180):
                try:child.wait(timeout=5);break
                except subprocess.TimeoutExpired:
                    if i%6==0:print('Waiting for atomic progress / immediate HF flush attempt. Keep session files.',flush=True)
        raise
    assert result==0,'Preserve NB13_RECOVERY diagnostics and run/console.log. No different model/experiment was substituted.'
print('Original NB13 recovery/resume finished. Save this executed notebook for HF verification.')
if (WORK/'LAST_REMOTE.json').exists():print(json.dumps(read(WORK/'LAST_REMOTE.json'),indent=2))


[NB13] verify inputs and prepare local-or-HF recovery
Checking the attached prepared dataset; it stays read-only.
[NB13 status] {"operation": "verify inputs and prepare local-or-HF recovery", "operation_seconds": 15.0, "local_cursor": {}, "free_bytes": 20936450048}
[NB13 status] {"operation": "verify inputs and prepare local-or-HF recovery", "operation_seconds": 30.0, "local_cursor": {}, "free_bytes": 20936445952}
[NB13 status] {"operation": "verify inputs and prepare local-or-HF recovery", "operation_seconds": 45.0, "local_cursor": {}, "free_bytes": 20936445952}
[NB13 status] {"operation": "verify inputs and prepare local-or-HF recovery", "operation_seconds": 60.0, "local_cursor": {}, "free_bytes": 20936445952}
[NB13 status] {"operation": "verify inputs and prepare local-or-HF recovery", "operation_seconds": 75.0, "local_cursor": {}, "free_bytes": 20936445952}
[NB13 status] {"operation": "verify inputs and prepare local-or-HF recovery", "operation_seconds": 90.0, "local_cursor": {}, "

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            
New Data Upload               : |          |  0.00B /  0.00B            

  ...acts/epochs/epoch_0051.pt:   0%|          | 19.4kB / 7.99MB            


  ...elected-identity-00032.pt: 100%|██████████| 1.00MB / 1.00MB            



  ...1733def28301/HISTORY.json:  21%|██        | 24.3MB /  116MB            

  ...acts/epochs/epoch_0051.pt:   0%|          | 19.4kB / 7.99MB            


  ...elected-identity-00032.pt: 100%|██████████| 1.00MB / 1.00MB            



Processing Files (1 / 53)     :   1%|          | 25.4MB / 2.04GB,   ???B/s  




  ...07/endpoint_attention.csv:  77%|███████▋  | 31.9MB / 41.3MB            





  ...s/selected/chunk_00032.pt:  13%|█▎        | 7.97MB / 61.8MB            






  ...elected-identity-00008.pt: 100%|██████████|  999kB /  999kB            







  ...s/selected/chunk_00024.pt:  13%|█▎        | 7.98MB / 61.8MB            








  ...s/selected/chunk_00040.pt:  13%|█▎   

[NB13 status] {"operation": "HF bounded commit / verification", "operation_seconds": 2.7, "local_cursor": {"seq": 3912, "updates": 2940, "phase": "completed"}, "free_bytes": 9610608640}




  ...acts/epochs/epoch_0051.pt:   0%|          | 19.4kB / 7.99MB            


  ...1733def28301/HISTORY.json:  31%|███▏      | 36.2MB /  116MB            



  ...s/selected/chunk_00032.pt:  65%|██████▍   | 39.9MB / 61.8MB            




  ...s/selected/chunk_00024.pt:  77%|███████▋  | 47.9MB / 61.8MB            





  ...s/selected/chunk_00040.pt:  78%|███████▊  | 47.9MB / 61.8MB            






  ...s/selected/chunk_00016.pt: 100%|██████████| 61.8MB / 61.8MB            







  ...s/selected/chunk_00008.pt: 100%|██████████| 61.8MB / 61.8MB            








  ...s/selected/chunk_00048.pt:  65%|██████▍   | 39.9MB / 61.8MB            









  ...elected-identity-00024.pt: 100%|██████████|  999kB /  999kB            










Processing Files (8 / 53)     :  20%|█▉        |  406MB / 2.04GB, 37.0MB/s  

  ...acts/epochs/epoch_0051.pt:   0%|          | 19.4kB / 7.99MB            


  ...1733def28301/HISTORY.json:  31%|███▏      | 36.2MB /  116MB            



  ...s/selecte

[NB13] HF bounded commit / verification
[NB13] HF repo_info
[NB13] HF resolve_json:OWNER.json
[NB13] HF resolve_json:BATCHES/9ef0bb139d906b61b612b6652dd30db109959187181b16c0d6c374d3798f8d51.json
[NB13 status] {"operation": "HF bounded commit / verification", "operation_seconds": 3.5, "local_cursor": {"seq": 3912, "updates": 2940, "phase": "completed"}, "free_bytes": 9610604544}
[NB13] HF bounded commit / verification
[NB13] HF repo_info
[NB13] HF resolve_json:OWNER.json
[NB13] HF resolve_json:BATCHES/76a4daee3fe8a21a2164702e7360f5cf138dd61e4840c95f49da1168ea76732f.json
[NB13 status] {"operation": "HF bounded commit / verification", "operation_seconds": 5.1, "local_cursor": {"seq": 3912, "updates": 2940, "phase": "completed"}, "free_bytes": 9610600448}
[NB13] HF bounded commit / verification
[NB13] HF repo_info
[NB13] HF resolve_json:OWNER.json
[NB13] HF resolve_json:BATCHES/2319327bd06c25ae223eda25671dbf5964e1b79518bc0911b1c22c6483d5ea56.json
[NB13] HF repo_info
[NB13] HF resolve_json: